In [ ]:
import os

# TO DO: manually edit file name, file type, worksheet name, and columns you'd use for geocoding a location.

FILE_NAME = "Landfills.csv"
MIME_TYPE = "text/csv"
WORKSHEET_NAME = "Landfills"
address_columns_sans_latlong = ["Landfill Name", "State", "Physical Address", "City", "County", "Zip Code"]
# Please do lat first, then long
latlong_columns_names = ["Latitude", "Longitude"]

# DO NOT CHANGE THESE.
ADDRESS_COLUMNS = address_columns_sans_latlong + latlong_columns_names
GSHEET_NAME = "address-to-geocoded" # look for this spreadsheet in Google Drive
geocoded_columns = ["status", "closest_address_line_1", "closest_address_line_2", "closest_city", "closest_county", "closest_state", "closest_postal_code", "closest_latitude", "closest_longitude", "closest_geoid", "closest_state_name", "closest_state_fips", "closest_county_name", "closest_county_fips","is_na","address_id", "address"]

# Determine project root for robust path discovery
PROJECT_ROOT = os.environ.get("CA_BIOSITING_PROJECT_ROOT", "../../../../../../")
CREDENTIALS_PATHS = [
    os.path.join(PROJECT_ROOT, "credentials.json"),
    os.path.join(PROJECT_ROOT, "resources/prefect/credentials.json"),
    "credentials.json"
]
CREDENTIALS_PATH = next((p for p in CREDENTIALS_PATHS if os.path.exists(p)), CREDENTIALS_PATHS[0])
DATASET_FOLDER = os.path.join(PROJECT_ROOT, "src/ca_biositing/pipeline/ca_biositing/pipeline/temp_external_datasets/")

In [ ]:
# TO DO: just run this code snippet. Extracts raw csv/zip/geojson file

import os
import pandas as pd
import gspread
from gspread.exceptions import WorksheetNotFound
from ca_biositing.pipeline.utils.gdrive_to_pandas import gdrive_to_df

print(f"Extracting raw data from '{FILE_NAME}'...")

credentials_path = CREDENTIALS_PATH
dataset_folder = DATASET_FOLDER

if os.path.exists(FILE_NAME):
    print(f"Found local file at '{FILE_NAME}'. Loading locally for testing...")
    raw_df = pd.read_csv(FILE_NAME)
else:
    print(f"Local file not found. Attempting to pull '{FILE_NAME}' from Google Drive using credentials at {CREDENTIALS_PATH}...")
    raw_df = gdrive_to_df(FILE_NAME, MIME_TYPE, credentials_path, dataset_folder)

if raw_df is None:
    print("Failed to extract data. Aborting.")

print("Successfully extracted raw data.")

In [ ]:
# TO DO: just run this code snippet.
# tests whether the worksheet exists; if not, it creates a new one
gc = gspread.service_account(filename=CREDENTIALS_PATH)
sh = gc.open(GSHEET_NAME)
try:
    worksheet = sh.worksheet(WORKSHEET_NAME)
except WorksheetNotFound:
    # Create new worksheet if it doesn't exist yet
    worksheet = sh.add_worksheet(title=WORKSHEET_NAME, rows=len(raw_df)+1, cols=len(geocoded_columns)+len(ADDRESS_COLUMNS))
    columns = [ADDRESS_COLUMNS+geocoded_columns]
    worksheet.update(columns)


# Geocoder Module

In [ ]:
# TO DO: just run this code snippet once.
from ca_biositing.pipeline.etl.extract.factory import create_extractor



extract = create_extractor(GSHEET_NAME, WORKSHEET_NAME)


In [ ]:
# TO DO: just run this code snippet.
df = extract(os.path.join(PROJECT_ROOT, "resources/prefect/"))

In [ ]:
df

In [ ]:
# TO DO: just run this code snippet.

# get unique rows from incoming dataset that aren't in google sheet
df_address = df[ADDRESS_COLUMNS]
raw_df_address = raw_df[ADDRESS_COLUMNS]

# This compares the incoming data to the data already in the Google Sheet
# And returns new rows
comparison_df = raw_df_address.merge(df_address.drop_duplicates(), on=ADDRESS_COLUMNS, 
                   how='left', indicator=True)
new_rows = comparison_df[comparison_df['_merge'] == 'left_only'][ADDRESS_COLUMNS]
new_rows["status"] = "pending"
new_rows
# add rows to google sheet df and list them as pending

In [ ]:
# TO DO: just run this code snippet. Concats new rows to df
df = pd.concat([df, new_rows])

In [ ]:
df

In [ ]:
# TO DO: just run this code snippet. geocodes addresses
from ca_biositing.pipeline.utils.geo_utils import parse_addresses
from dotenv import load_dotenv
import pandas as pd

load_dotenv(os.path.join(PROJECT_ROOT, "resources/docker/.env"))

# df = df.astype({'latitude': float, 'longitude': float})

address_df, geoid_df = parse_addresses(
        df,
        merge_columns=address_columns_sans_latlong,
        lat=latlong_columns_names[0],
        long=latlong_columns_names[1],
    )

added_address_df = pd.concat([address_df, geoid_df], axis=1)
added_address_df

In [ ]:
# TO DO: just run this code snippet. adds geocoded data to df
replace_columns = added_address_df.columns
df.loc[df["status"] == "pending", replace_columns] = added_address_df
df['closest_county_name'] = df['closest_county_name'].str.replace(" County", "")

In [ ]:
# TO DO: just run this code snippet. finds addresses marked "true" to load into LocationAddress database
found_addresses = df[df["status"] == "true"]
found_addresses

In [ ]:
# TO DO: just run this code snippet. Loads addresses and FIPS codes to databases

print("Bridging County (Place) to LocationAddress...")
from sqlmodel import Session, select, create_engine
from ca_biositing.pipeline.utils.engine import _get_database_url, get_engine
from ca_biositing.datamodels.models import LocationAddress, Place
from ca_biositing.datamodels.config import settings

db_url = _get_database_url()
db_url = db_url.replace("@db:5432", f"@localhost:{settings.POSTGRES_PORT}")
db_url = db_url.replace("@db:", f"@localhost:{settings.POSTGRES_PORT}")
engine = create_engine(
        db_url,
        pool_size=5,
        max_overflow=0,
        pool_pre_ping=True,
        connect_args={"connect_timeout": 10}
)

with Session(engine) as session:
    place_to_address_map = {}

    for index, row in found_addresses.iterrows():
        geoid = row.get("closest_geoid")
        if geoid != 00000 and geoid is not None:
            stmt1 = select(Place).where(Place.geoid == geoid)
            place = session.exec(stmt1).first()

            stmt2 = select(LocationAddress).where(
                LocationAddress.geography_id == geoid
            )
            address = session.exec(stmt2).first()

            if not place:
                place = Place(
                    geoid=geoid,
                    state_name=row.get("closest_state_name"),
                    state_fips=row.get("closest_state_fips"),
                    county_name=row.get("closest_county_name"),
                    county_fips=row.get("closest_county_fips"),
                )
                session.add(place)
                session.flush()

            if not address:
                address = LocationAddress(
                    geography_id=geoid,
                    address_line1=row.get("closest_address_line_1"),
                    address_line2=row.get("closest_address_line_2"),
                    city=row.get("closest_city"),
                    zip=row.get("closest_postal_code"),
                    lat=row.get("closest_latitude"),
                    lon=row.get("closest_longitude"),
                    is_anonymous=False,
                )
                session.add(address)
                session.flush()

            place_to_address_map[geoid] = address.id

    session.commit()
    found_addresses["address_id"] = found_addresses["closest_geoid"].map(
        place_to_address_map
    )
    print(
        f"Mapped {len(place_to_address_map)} counties to LocationAddresses"
    )

In [ ]:
# TO DO: just run this code snippet. Sets address_ids to ids assigned in database
df['address_id'] = 0
df.loc[df["status"] == "true", "address_id"] = found_addresses["address_id"]
df = df.drop_duplicates(keep="first")

In [ ]:
# TO DO: just run this code snippet. Adds df to the Google Sheet. Please check to make sure changes are okay.

df = df.fillna("")
worksheet = sh.worksheet(WORKSHEET_NAME)
worksheet.update([df.columns.values.tolist()] + df.values.tolist())

# Visualize Results

In [ ]:
import folium

map_center = [37.5, -119.5] # Central California
m = folium.Map(location=map_center, zoom_start=6)

for _, row in df[df['status'] == 'true'].iterrows():
    folium.Marker(
        location=[row['closest_latitude'], row['closest_longitude']],
        popup=f"{row['closest_address_line_1']}<br>Status: {row['status']}",
        icon=folium.Icon(color='green')
    ).add_to(m)

for _, row in df[df['status'] == 'false'].iterrows():
    if row['closest_latitude'] != 0:
        folium.Marker(
            location=[row['closest_latitude'], row['closest_longitude']],
            popup="Geocoding Failed",
            icon=folium.Icon(color='red', icon='info-sign')
        ).add_to(m)

m